# Optymalizacja Lokalizacji Nowej Firmy przy Użyciu Algorytmu AAIA

Ten notebook demonstruje wykorzystanie algorytmu Artificial Afterimage Algorithm (AAIA) do optymalizacji lokalizacji nowej firmy na podstawie danych o istniejących firmach w regionie. Skupia się na klasteryzacji istniejących firm oraz bezpośredniej optymalizacji lokalizacji nowej firmy.

## Import Required Libraries

Import necessary libraries including NumPy, Pandas, Matplotlib, Scikit-learn, and the AAIA module for clustering and optimization.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances
import sys
import os

# Add path to aaia module
sys.path.append(os.path.join(os.getcwd(), '..'))
import aaia


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\dusza\anaconda3\envs\abd_env\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\dusza\anaconda3\envs\abd_env\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\dusza\anaconda3\envs\abd_env\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\dusza\anaconda3\envs\abd_env\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.s

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\dusza\anaconda3\envs\abd_env\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\dusza\anaconda3\envs\abd_env\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\dusza\anaconda3\envs\abd_env\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\dusza\anaconda3\envs\abd_env\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.s

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\dusza\anaconda3\envs\abd_env\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\dusza\anaconda3\envs\abd_env\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\dusza\anaconda3\envs\abd_env\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\dusza\anaconda3\envs\abd_env\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.s

AttributeError: _ARRAY_API not found

ImportError: initialization failed

## Generate Synthetic Business Dataset

Create a synthetic dataset of existing businesses with features: business ID, type (category/PKD), geographic coordinates (latitude, longitude), region, number of competitors within radius, population density, accessibility score, and rental cost. Simulate businesses distributed across a region with varying concentrations.

In [ ]:
np.random.seed(42)

# Define region bounds (e.g., Krakow area)
lat_min, lat_max = 49.9, 50.1
lon_min, lon_max = 19.8, 20.2

# Business types
business_types = ['restaurant', 'gym', 'workshop', 'grocery_store']

# Generate businesses
n_businesses = 200
businesses = []

for i in range(n_businesses):
    lat = np.random.uniform(lat_min, lat_max)
    lon = np.random.uniform(lon_min, lon_max)
    business_type = np.random.choice(business_types)
    
    # Simulate population density (higher in center)
    center_lat, center_lon = 50.0, 20.0
    distance_to_center = np.sqrt((lat - center_lat)**2 + (lon - center_lon)**2)
    pop_density = max(0, 1000 - distance_to_center * 10000)  # Higher near center
    
    # Competitors within 1km (simplified)
    competitors = np.random.poisson(5 * (1 - distance_to_center * 10))
    
    # Accessibility score (higher near roads, simulated)
    accessibility = np.random.uniform(0.5, 1.0) + 0.3 * (1 - distance_to_center * 5)
    accessibility = min(1.0, accessibility)
    
    # Rental cost (higher in center)
    rental_cost = 50 + distance_to_center * 200 + np.random.normal(0, 10)
    
    businesses.append({
        'id': i,
        'type': business_type,
        'lat': lat,
        'lon': lon,
        'pop_density': pop_density,
        'competitors': competitors,
        'accessibility': accessibility,
        'rental_cost': rental_cost
    })

df_businesses = pd.DataFrame(businesses)
print(f"Generated {len(df_businesses)} businesses")
print(df_businesses.head())

## Prepare Data for AAIA Algorithm

Extract features relevant for location optimization: coordinates, competitor density, population metrics, and accessibility. Normalize features to ensure proper scaling. Structure data as feature vectors that represent potential business locations as candidates for evaluation.

In [ ]:
# Select features for optimization
features = ['lat', 'lon', 'pop_density', 'competitors', 'accessibility', 'rental_cost']
X = df_businesses[features].values

# Normalize features
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)

print(f"Feature matrix shape: {X_normalized.shape}")
print("Features:", features)

## Define Cost Function for Location Optimization

Implement an objective function that evaluates candidate locations for new businesses. The function should balance: closeness to customers (high population density), distance from competition (low competitor count nearby), accessibility (proximity to roads/transport), and operational costs. Return a single score to maximize for optimal location.

In [ ]:
def objective_function(candidate, data, weights=None):
    """
    Objective function for location optimization.
    J(x,y) = a * pop_density - b * competitors + c * accessibility - d * rental_cost
    We maximize J, so for minimization in AAIA, return -J
    """
    if weights is None:
        weights = {'pop_density': 1.0, 'competitors': 0.5, 'accessibility': 0.8, 'rental_cost': 0.3}
    
    # candidate is [lat, lon, pop_density, competitors, accessibility, rental_cost]
    # But for new location, we need to estimate pop_density, competitors, etc. based on location
    # For simplicity, assume candidate represents the location features directly
    # In real scenario, we'd interpolate from data
    
    lat, lon, pop_density, competitors, accessibility, rental_cost = candidate
    
    J = (weights['pop_density'] * pop_density - 
         weights['competitors'] * competitors + 
         weights['accessibility'] * accessibility - 
         weights['rental_cost'] * rental_cost)
    
    return -J  # Minimize -J to maximize J

# For AAIA, we need to adapt it to maximize objective
# We'll modify the fitness to use our objective function

## Run AAIA Algorithm for Location Optimization

Use the AAIA algorithm to search for the optimal new business location. Set parameters: initial population of candidate locations, maximum iterations, and convergence criteria. Track the best solution found and visualize the evolution of the objective function across iterations.

In [ ]:
def calculate_best_and_worst_location(population, data, objective_func):
    scores = np.array([objective_func(candidate, data) for candidate in population])
    best = population[np.argmin(scores)]  # Minimize objective
    worst = population[np.argmax(scores)]
    return best, worst

def find_optimal_location(data, objective_func, max_iterations=1000, population_size=50):
    """
    Find optimal location using AAIA-inspired algorithm with custom objective function.
    """
    count = 0
    low = data.min(axis=0)
    high = data.max(axis=0)
    population = np.random.uniform(low, high, size=(population_size, data.shape[1]))
    best_solution, worst_solution = calculate_best_and_worst_location(population, data, objective_func)

    fitness_history = []

    while count < max_iterations:
        V = aaia.calculate_visual_angle(population, best_solution)
        S = aaia.calculate_perceptual_size(V, population, best_solution)

        local_best, local_worst = calculate_best_and_worst_location(population, data, objective_func)

        if objective_func(local_best, data) < objective_func(best_solution, data):
            best_solution = local_best
        if objective_func(local_worst, data) > objective_func(worst_solution, data):
            worst_solution = local_worst

        population = aaia.calculate_population(S, best_solution, worst_solution, population)
        population = np.clip(population, low, high)
        
        fitness_history.append(objective_func(best_solution, data))
        count += 1

    return best_solution, fitness_history

# Run the algorithm
np.random.seed(123)
optimal_location, history = find_optimal_location(X_normalized, objective_function, max_iterations=500, population_size=30)

print("Optimal location (normalized):", optimal_location)
print("Final objective value:", objective_function(optimal_location, X_normalized))

# Inverse transform to get actual coordinates
optimal_actual = scaler.inverse_transform(optimal_location.reshape(1, -1))[0]
print("Optimal location (actual):", optimal_actual)

## Evaluate and Visualize Results

Visualize the region map with existing businesses, competitor clusters, population density heatmap, and the recommended optimal location for new business. Generate performance metrics: accuracy of clustering, distance from nearest competitor, population within radius, and predicted foot traffic. Create comparison charts showing different business types and their optimal locations.

In [ ]:
# Plot the results
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Business locations by type
ax = axes[0, 0]
colors = {'restaurant': 'red', 'gym': 'blue', 'workshop': 'green', 'grocery_store': 'orange'}
for typ in business_types:
    subset = df_businesses[df_businesses['type'] == typ]
    ax.scatter(subset['lon'], subset['lat'], c=colors[typ], label=typ, alpha=0.6)
ax.scatter(optimal_actual[1], optimal_actual[0], c='black', marker='*', s=200, label='Optimal Location')
ax.set_title('Business Locations and Optimal New Location')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend()
ax.grid(True)

# 2. Population density heatmap
ax = axes[0, 1]
sc = ax.scatter(df_businesses['lon'], df_businesses['lat'], c=df_businesses['pop_density'], cmap='viridis', alpha=0.6)
ax.scatter(optimal_actual[1], optimal_actual[0], c='red', marker='*', s=200, label='Optimal')
plt.colorbar(sc, ax=ax, label='Population Density')
ax.set_title('Population Density')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True)

# 3. Competitors heatmap
ax = axes[1, 0]
sc = ax.scatter(df_businesses['lon'], df_businesses['lat'], c=df_businesses['competitors'], cmap='Reds', alpha=0.6)
ax.scatter(optimal_actual[1], optimal_actual[0], c='blue', marker='*', s=200, label='Optimal')
plt.colorbar(sc, ax=ax, label='Competitors Count')
ax.set_title('Competitor Density')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True)

# 4. Fitness history
ax = axes[1, 1]
ax.plot(history)
ax.set_title('Objective Function Evolution')
ax.set_xlabel('Iteration')
ax.set_ylabel('Objective Value (minimized)')
ax.grid(True)

plt.tight_layout()
plt.show()

# Print metrics
print(f"Optimal Location: Lat={optimal_actual[0]:.4f}, Lon={optimal_actual[1]:.4f}")
print(f"Population Density: {optimal_actual[2]:.2f}")
print(f"Competitors: {optimal_actual[3]:.2f}")
print(f"Accessibility: {optimal_actual[4]:.2f}")
print(f"Rental Cost: {optimal_actual[5]:.2f}")

# Distance to nearest competitor
distances = euclidean_distances([[optimal_actual[0], optimal_actual[1]]], df_businesses[['lat', 'lon']].values)
min_distance = np.min(distances)
print(f"Distance to nearest business: {min_distance:.4f} degrees")